# DistilBERT baseline on GPU (real AG News data)

Three epoch GPU experiment using the same prepared 8,000 training and 2,000 validation examples. The original test split is not used for checkpoint selection.

GPU setup on Windows: from the project root, run .venv/Scripts/python.exe -m pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu132. Verify torch.cuda.is_available() before running. The previous CPU model and its test results remain separate.

In [1]:
from pathlib import Path
import torch

from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ag_news'
MODEL_DIR = PROJECT_ROOT / 'models' / 'distilbert_ag_news_baseline_gpu'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'distilbert_ag_news_baseline_gpu'
SEED = 42
MODEL_NAME = 'distilbert/distilbert-base-uncased'
assert torch.cuda.is_available(), 'CUDA GPU required for this notebook'
print('GPU:', torch.cuda.get_device_name(0))
set_seed(SEED)

splits = load_from_disk(str(DATA_DIR))
label_names = splits['train'].features['label'].names
assert len(splits['train']) == 8_000
assert len(splits['validation']) == 2_000
assert len(splits['test']) == 7_600
assert len(label_names) == 4
print({name: len(split) for name, split in splits.items()})
print('Labels:', label_names)

E:\VGTU\5 semester\Natural Language Processing\lab 4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 4060 Laptop GPU


{'train': 8000, 'validation': 2000, 'test': 7600}
Labels: ['World', 'Sports', 'Business', 'Sci/Tech']


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={index: name for index, name in enumerate(label_names)},
    label2id={name: index for index, name in enumerate(label_names)},
)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

tokenized = splits.select_columns(['text', 'label']).map(
    tokenize, batched=True, remove_columns=['text']
)
print(tokenized)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


Loading weights:   3%|▎         | 3/100 [00:00<00:04, 23.62it/s]


Loading weights:  11%|█         | 11/100 [00:00<00:01, 53.24it/s]


Loading weights:  17%|█▋        | 17/100 [00:00<00:02, 36.07it/s]


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 203.20it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})


In [ ]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=250,
    save_strategy='steps',
    save_steps=250,
    logging_strategy='steps',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
    bf16=True,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)
print('Training examples:', len(trainer.train_dataset))
print('Validation examples:', len(trainer.eval_dataset))
print('Checkpoints:', CHECKPOINT_DIR)

Training examples: 8000
Validation examples: 2000
Checkpoints: E:\VGTU\5 semester\Natural Language Processing\lab 4\checkpoints\distilbert_ag_news_baseline_gpu


In [4]:
train_result = trainer.train()
print('Training loss:', train_result.training_loss)
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best validation loss:', trainer.state.best_metric)

Step,Training Loss,Validation Loss
250,0.379027,0.340689
500,0.283202,0.299852
750,0.191913,0.330152
1000,0.186055,0.330501
1250,0.087381,0.365442
1500,0.076377,0.367196



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]

Training loss: 0.20920643933614094
Best checkpoint: E:\VGTU\5 semester\Natural Language Processing\lab 4\checkpoints\distilbert_ag_news_baseline_gpu\checkpoint-500
Best validation loss: 0.2998521625995636


In [5]:
import json

MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
configuration = {
    'source_dataset': 'fancyzhx/ag_news',
    'prepared_data': str(DATA_DIR),
    'model_name': MODEL_NAME,
    'seed': SEED,
    'device': torch.cuda.get_device_name(0),
    'precision': 'bf16',
    'training_samples': len(splits['train']),
    'validation_samples': len(splits['validation']),
    'test_samples_unused': len(splits['test']),
    'label_names': label_names,
    'max_length': 128,
    'num_train_epochs': training_args.num_train_epochs,
    'train_batch_size': training_args.per_device_train_batch_size,
    'eval_batch_size': training_args.per_device_eval_batch_size,
    'learning_rate': training_args.learning_rate,
    'weight_decay': training_args.weight_decay,
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_loss': trainer.state.best_metric,
}
(MODEL_DIR / 'training_config.json').write_text(
    json.dumps(configuration, indent=2), encoding='utf-8'
)
(MODEL_DIR / 'training_history.json').write_text(
    json.dumps(trainer.state.log_history, indent=2), encoding='utf-8'
)
print('Saved best model and training configuration to', MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.32it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.32it/s]

Saved best model and training configuration to E:\VGTU\5 semester\Natural Language Processing\lab 4\models\distilbert_ag_news_baseline_gpu


## Run result

Validation loss was lowest at step 500 (0.2999). It rose at later checkpoints, reaching 0.3672 at step 1500, so the notebook exported the step 500 weights. The longer run did not improve validation loss over the earlier CPU baseline (0.2947); additional epochs alone were not beneficial for this configuration.